In [1]:
"""
BB84 Quantum Key Distribution simulation with Qiskit.

Run:  python bb84.py
Each qubit is sent exactly once (1 shot), like a real channel.
Basis encoding: 0 = Z basis (|0>,|1>), 1 = X basis (|+>,|->)
"""
import random
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
STATES = {(0, 0): "|0>", (1, 0): "|1>", (0, 1): "|+>", (1, 1): "|->"}


def encode(bit, basis):
    """Alice's preparation: bit=1 -> X gate; basis=X -> H gate."""
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc


def measure(qc, basis):
    """Measure in Z (plain) or X (apply H first)."""
    if basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    return qc


def run_all(circuits):
    """Run each circuit for a single shot; return list of measured bits."""
    job = sim.run(transpile(circuits, sim), shots=1)
    counts = job.result().get_counts()
    if isinstance(counts, dict):
        counts = [counts]
    return [int(list(c.keys())[0]) for c in counts]


def bb84(n_qubits=200, eve=False, sample_fraction=0.5, seed=None):
    if seed is not None:
        random.seed(seed)

    # 1. Alice: random bits and random bases
    a_bits = [random.randint(0, 1) for _ in range(n_qubits)]
    a_bases = [random.randint(0, 1) for _ in range(n_qubits)]
    circuits = [encode(b, bs) for b, bs in zip(a_bits, a_bases)]

    # 2. (Optional) Eve intercepts, measures in a random basis, resends
    e_bases = e_bits = None
    if eve:
        e_bases = [random.randint(0, 1) for _ in range(n_qubits)]
        e_bits = run_all([measure(qc, bs) for qc, bs in zip(circuits, e_bases)])
        circuits = [encode(b, bs) for b, bs in zip(e_bits, e_bases)]

    # 3. Bob: random bases, measure
    b_bases = [random.randint(0, 1) for _ in range(n_qubits)]
    b_bits = run_all([measure(qc, bs) for qc, bs in zip(circuits, b_bases)])

    # 4. Sifting: keep positions where bases matched
    keep = [i for i in range(n_qubits) if a_bases[i] == b_bases[i]]
    alice_key = [a_bits[i] for i in keep]
    bob_key = [b_bits[i] for i in keep]

    # 5. Eavesdropper check: publicly compare a random sample
    k = max(1, int(len(keep) * sample_fraction))
    sample_idx = set(random.sample(range(len(keep)), k))
    errors = sum(alice_key[i] != bob_key[i] for i in sample_idx)
    error_rate = errors / k

    # 6. Remaining bits (sample discarded) = final secret key
    final_a = [alice_key[i] for i in range(len(keep)) if i not in sample_idx]
    final_b = [bob_key[i] for i in range(len(keep)) if i not in sample_idx]

    # Per-qubit record of everything that happened
    table = []
    for i in range(n_qubits):
        table.append({
            "n": i,
            "a_bit": a_bits[i], "a_basis": "ZX"[a_bases[i]],
            "state": STATES[(a_bits[i], a_bases[i])],
            "e_basis": "ZX"[e_bases[i]] if eve else "-",
            "e_bit": e_bits[i] if eve else "-",
            "b_basis": "ZX"[b_bases[i]], "b_bit": b_bits[i],
            "keep": a_bases[i] == b_bases[i],
            "match": a_bits[i] == b_bits[i],
        })

    return {
        "table": table, "sent": n_qubits, "sifted": len(keep),
        "sample_size": k, "errors": errors, "error_rate": error_rate,
        "final_len": len(final_a), "keys_match": final_a == final_b,
        "final_key_alice": final_a, "final_key_bob": final_b,
    }


def print_table(r, limit=20, eve=False):
    """Print per-qubit outputs for the first `limit` qubits."""
    print(f"\nPer-qubit outputs (first {limit} of {r['sent']}):")
    hdr = f"{'#':>3} | {'Alice bit':>9} {'basis':>5} {'state':>5} |"
    if eve:
        hdr += f" {'Eve basis':>9} {'Eve bit':>7} |"
    hdr += f" {'Bob basis':>9} {'Bob bit':>7} | {'Keep?':>5} {'Match?':>6}"
    print(hdr)
    print("-" * len(hdr))
    for t in r["table"][:limit]:
        row = f"{t['n']:>3} | {t['a_bit']:>9} {t['a_basis']:>5} {t['state']:>5} |"
        if eve:
            row += f" {t['e_basis']:>9} {str(t['e_bit']):>7} |"
        row += f" {t['b_basis']:>9} {t['b_bit']:>7} | "
        row += f"{'yes' if t['keep'] else 'no':>5} "
        row += f"{('yes' if t['match'] else 'NO') if t['keep'] else '-':>6}"
        print(row)


def show_example_circuit():
    """Draw the circuit for one qubit: Alice sends bit=1 in X basis, Bob measures in X."""
    qc = encode(1, 1)
    qc.barrier()
    qc = measure(qc, 1)
    print("\nExample circuit (Alice: bit=1, X basis -> |->; Bob measures in X):")
    print(qc.draw("text"))


def report(title, r):
    print(f"\n=== {title} ===")
    print(f"Qubits sent:            {r['sent']}")
    print(f"After sifting:          {r['sifted']}  (~50% expected)")
    print(f"Bits compared (public): {r['sample_size']}")
    print(f"Errors found:           {r['errors']}")
    print(f"Error rate:             {r['error_rate']:.1%}")
    verdict = ("EAVESDROPPER DETECTED -> discard key" if r["error_rate"] > 0.11
               else "Channel looks clean -> keep key")
    print(f"Verdict:                {verdict}")
    print(f"Final key length:       {r['final_len']}")
    print(f"Final keys identical:   {r['keys_match']}")
    print(f"Alice key (first 32):   {''.join(map(str, r['final_key_alice'][:32]))}")
    print(f"Bob key   (first 32):   {''.join(map(str, r['final_key_bob'][:32]))}")


if __name__ == "__main__":
    show_example_circuit()

    r = bb84(n_qubits=400, eve=False, seed=1)
    print("\n" + "#" * 20 + " WITHOUT EVE " + "#" * 20)
    print_table(r, limit=20, eve=False)
    report("Without Eve", r)

    r = bb84(n_qubits=400, eve=True, seed=1)
    print("\n" + "#" * 20 + " WITH EVE " + "#" * 20)
    print_table(r, limit=20, eve=True)
    report("With Eve (intercept-resend)", r)

    runs = 20
    avg = sum(bb84(200, eve=True)["error_rate"] for _ in range(runs)) / runs
    print(f"\nAverage error rate with Eve over {runs} runs: {avg:.1%}  (theory: 25%)")


Example circuit (Alice: bit=1, X basis -> |->; Bob measures in X):
     ┌───┐┌───┐ ░ ┌───┐┌─┐
  q: ┤ X ├┤ H ├─░─┤ H ├┤M├
     └───┘└───┘ ░ └───┘└╥┘
c: 1/═══════════════════╩═
                        0 

#################### WITHOUT EVE ####################

Per-qubit outputs (first 20 of 400):
  # | Alice bit basis state | Bob basis Bob bit | Keep? Match?
--------------------------------------------------------------
  0 |         0     Z   |0> |         X       0 |    no      -
  1 |         0     Z   |0> |         Z       0 |   yes    yes
  2 |         1     X   |-> |         Z       0 |    no      -
  3 |         0     X   |+> |         X       0 |   yes    yes
  4 |         1     X   |-> |         Z       0 |    no      -
  5 |         1     X   |-> |         Z       1 |    no      -
  6 |         1     Z   |1> |         X       0 |    no      -
  7 |         1     X   |-> |         Z       0 |    no      -
  8 |         0     Z   |0> |         Z       0 |   yes    yes
  9 |      